# YOLO11n-Pose Training on LV-MHP-v2

Finetune YOLO11n-pose on LV-MHP-v2 dataset with 16 body keypoints.

**Setup:** Runtime > Change runtime type > **GPU (T4)** or **TPU**

## 1. Install Dependencies

In [ ]:
!pip install ultralytics -q

## 2. Check Hardware

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Check for TPU
try:
    import torch_xla.core.xla_model as xm
    print(f"TPU available: {xm.xla_device()}")
except ImportError:
    print("TPU: not available (GPU runtime selected)")

## 3. Mount Google Drive

Upload your `LV-MHP-v2-pose` folder to Google Drive before running this cell.

Expected structure on Drive:
```
My Drive/
  PointsX/
    LV-MHP-v2-pose/
      train/
        images/
        labels/
      val/
        images/
        labels/
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# CONFIGURE: Set your Google Drive path here
# ============================================================
DRIVE_DATASET_DIR = "/content/drive/MyDrive/PointsX/LV-MHP-v2-pose"

# Verify dataset exists
import os
for split in ["train", "val"]:
    imgs = os.path.join(DRIVE_DATASET_DIR, split, "images")
    lbls = os.path.join(DRIVE_DATASET_DIR, split, "labels")
    n_imgs = len(os.listdir(imgs)) if os.path.isdir(imgs) else 0
    n_lbls = len(os.listdir(lbls)) if os.path.isdir(lbls) else 0
    print(f"{split}: {n_imgs} images, {n_lbls} labels")

## 4. Copy Dataset to Local Storage (Faster I/O)

Training directly from Google Drive is slow. Copy to Colab's local SSD first.

In [ ]:
LOCAL_DATASET_DIR = "/content/LV-MHP-v2-pose"

if not os.path.exists(LOCAL_DATASET_DIR):
    print("Copying dataset to local storage (this takes a few minutes)...")
    !cp -r "{DRIVE_DATASET_DIR}" "{LOCAL_DATASET_DIR}"
    print("Done!")
else:
    print("Dataset already copied locally.")

# Verify
for split in ["train", "val"]:
    n = len(os.listdir(f"{LOCAL_DATASET_DIR}/{split}/images"))
    print(f"{split}: {n} images")

## 5. Create Dataset YAML

In [ ]:
DATASET_YAML = "/content/lv-mhp-v2-pose.yaml"

yaml_content = f"""# LV-MHP-v2 Pose Dataset - 16 body keypoints
path: {LOCAL_DATASET_DIR}
train: train/images
val: val/images

kpt_shape: [16, 3]

# Left-right flip indices for augmentation
flip_idx: [5, 4, 3, 2, 1, 0, 6, 7, 8, 9, 15, 14, 13, 12, 11, 10]

names:
  0: person
"""

with open(DATASET_YAML, "w") as f:
    f.write(yaml_content)

print(f"Dataset YAML written to: {DATASET_YAML}")
print(yaml_content)

## 6. Train YOLO11n-Pose

In [ ]:
from ultralytics import YOLO

# ============================================================
# CONFIGURE: Training hyperparameters
# ============================================================
EPOCHS = 100
BATCH_SIZE = -1  # auto (maximizes GPU memory usage)
IMG_SIZE = 640

model = YOLO("yolo11n-pose.pt")  # downloads automatically

results = model.train(
    data=DATASET_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    project="/content/runs",
    name="pose",
    exist_ok=True,
    pretrained=True,
    patience=20,
    workers=2,
    device=0,  # GPU 0
)

## 7. View Results

In [ ]:
from IPython.display import Image, display

# Training curves
display(Image(filename="/content/runs/pose/results.png", width=800))

In [ ]:
# Validation predictions
display(Image(filename="/content/runs/pose/val_batch0_pred.jpg", width=800))

In [ ]:
# Confusion matrix
display(Image(filename="/content/runs/pose/confusion_matrix.png", width=600))

## 8. Validate Best Model

In [ ]:
best_model = YOLO("/content/runs/pose/weights/best.pt")
metrics = best_model.val(data=DATASET_YAML)

print(f"\nBox mAP50: {metrics.box.map50:.4f}")
print(f"Box mAP50-95: {metrics.box.map:.4f}")
print(f"Pose mAP50: {metrics.pose.map50:.4f}")
print(f"Pose mAP50-95: {metrics.pose.map:.4f}")

## 9. Test Inference

In [ ]:
import glob

# Run inference on a few validation images
val_images = sorted(glob.glob(f"{LOCAL_DATASET_DIR}/val/images/*.jpg"))[:4]
results = best_model(val_images, imgsz=640)

for r in results:
    im = r.plot()
    display(Image(data=r.save(filename=None)))

## 10. Save Best Model to Google Drive

In [ ]:
import shutil

DRIVE_OUTPUT = "/content/drive/MyDrive/PointsX/runs/pose"
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Copy best and last weights
for name in ["best.pt", "last.pt"]:
    src = f"/content/runs/pose/weights/{name}"
    dst = f"{DRIVE_OUTPUT}/{name}"
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"Saved: {dst}")

# Copy results
for name in ["results.csv", "results.png"]:
    src = f"/content/runs/pose/{name}"
    dst = f"{DRIVE_OUTPUT}/{name}"
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"Saved: {dst}")

print("\nAll results saved to Google Drive!")